In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import os

In [ ]:
# 성연령별 2020 ~ 2023
merged_sex_age = gpd.read_file('../P1_Data/02.Pretreatment/merged_sex_age(20~23).shp',encoding='cp949')
# 시간별 2020 ~ 2023
merged_time = gpd.read_file('../P1_Data/02.Pretreatment/merged_time(20~23).shp',encoding='cp949')
# 요일별 2020 ~ 2023
merged_day = gpd.read_file('../P1_Data/02.Pretreatment/merged_day(20~23).shp',encoding='cp949')

### 1. 생활인구 패턴 분석
####    가. 시흥시 생활인구 패턴
#####       1) 전년 동월 대비 생활인구 비교
#####       2) 시간대별 생활인구 비교
#####       3) 요일별 생활인구 비교
#####       3) 성별 생활인구 비교

In [ ]:
#복사
pop_gender_age = merged_sex_age.copy()
pop_time = merged_time.copy()
pop_day = merged_day.copy()

In [ ]:
# etl_ym 열에서 년과 월을 추출합니다.
### (성연령별)
pop_gender_age['Year'] = pop_gender_age['etl_ym'] // 100
pop_gender_age['Month'] = pop_gender_age['etl_ym'] % 100

### (시간별)
pop_time['Year'] = pop_time['etl_ym'] // 100
pop_time['Month'] = pop_time['etl_ym'] % 100

### (요일별)
pop_day['Year'] = pop_day['etl_ym'] // 100
pop_day['Month'] = pop_day['etl_ym'] % 100

In [ ]:
pop_gender = pop_gender_age.copy()
# 성연령별 생활인구 컬럼 이름을 리스트로 저장합니다.
gender_age = ['M00_09', 'M10_19', 'M20_29', 'M30_39','M40_49', 'M50_59', 'M60_up', 'F00_09', 'F10_19', 'F20_29', 'F30_39','F40_49', 'F50_59', 'F60_up']

# 성연령별 컬럼 값을 가로로 더해서 'Total' 컬럼에 추가합니다.
pop_gender['생활인구'] = pop_gender[gender_age].sum(axis=1)
pop_gender

In [ ]:
#시흥시 생활인구 패턴_시흥시 공주시 인구비교_2022년 12월기준
data_2022 = pop_gender[pop_gender['Year'] == 2022.0]
data_2022
#월평균
grouped_data  = data_2022.groupby(['Year', 'Month'])['생활인구'].sum().round(1)
# 'total' 컬럼의 값의 전체 평균을 계산합니다.
total_mean = grouped_data.mean()
total_mean

In [ ]:
#시흥시 생활인구 패턴_시흥시 거주인구 및 생활인구 비교
tot_pop_gender_age = pop_gender.copy()
tot_pop_gender_age = tot_pop_gender_age[['etl_ym','Year' ,'Month','생활인구']]

tot_pop_gender_age = tot_pop_gender_age.groupby(['etl_ym','Year','Month']).sum().round(1).reset_index()
tot_pop_gender_age['생활인구'] = (tot_pop_gender_age['생활인구'] / 30).round(1)

## 연도별 월평균 생활인구 및 주민등록인구 비교
region_pop = pd.read_csv('../P1_Data/01.External/kosis/주민등록인구_시도_시_군_구__20230919183131.csv',encoding='cp949')

result = pd.concat([tot_pop_gender_age, region_pop[['계 (명)']]], axis=1)
result = result.rename(columns={'계 (명)' : '주민등록인구'})

# 연도와 월을 기준으로 그룹화하고 월평균을 계산
result2 = result.groupby(['Year'])[['생활인구', '주민등록인구']].mean().round(1).reset_index()
result2.to_csv('../P3_Result/01.CSV/1.시흥시_생활인구패턴/전체연도_월평균_생활인구_주민등록인구비교.csv',encoding='cp949',index=False)
result2

In [ ]:
#시흥시 생활인구 패턴 _ 전년동원대비 총 생활인구 비교
#시흥시 생활인구 패턴 _ 성별 생활인구

pop_gender = pop_gender_age.copy()
# 성연령별 생활인구 컬럼 이름을 리스트로 저장합니다.
gender_age = ['M00_09', 'M10_19', 'M20_29', 'M30_39','M40_49', 'M50_59', 'M60_up', 'F00_09', 'F10_19', 'F20_29', 'F30_39','F40_49', 'F50_59', 'F60_up']
# 성별 생활인구 컬럼 이름을 리스트로 저장합니다
female = ['F00_09', 'F10_19', 'F20_29','F30_39', 'F40_49', 'F50_59', 'F60_up']
male = ['M00_09', 'M10_19', 'M20_29','M30_39', 'M40_49', 'M50_59', 'M60_up']


# 성연령별 컬럼 값을 가로로 더해서 'Total' 컬럼에 추가합니다.
pop_gender['생활인구'] = pop_gender[gender_age].sum(axis=1)
# 남자
pop_gender['생활인구_남'] = pop_gender[male].sum(axis=1)
# 여자
pop_gender['생활인구_여'] = pop_gender[female].sum(axis=1)

tot_pop_gender_age = pop_gender.copy()
tot_pop_gender_age = tot_pop_gender_age[['etl_ym','Year','Month', '생활인구','생활인구_남','생활인구_여']]


In [ ]:
tot_pop_gender_age = tot_pop_gender_age.groupby(['etl_ym','Year','Month']).sum().round(1).reset_index()

#tot_pop_gender_age['생활인구'] = (tot_pop_gender_age['생활인구'] / 30).round(1)
#tot_pop_gender_age['생활인구_남'] = (tot_pop_gender_age['생활인구_남'] / 30).round(1)
#tot_pop_gender_age['생활인구_여'] = (tot_pop_gender_age['생활인구_여'] / 30).round(1)

tot_pop_gender_age.to_csv('../P3_Result/01.CSV/1.시흥시_생활인구패턴/전체연도_월별_성별_총생활인구_0922.csv',index=False ,encoding='cp949')
tot_pop_gender_age

In [ ]:
tot_sx_pop_gender_age = tot_pop_gender_age.groupby(['Year']).mean().round(1).reset_index()
tot_sx_pop_gender_age['생활인구_남_비율'] = (tot_sx_pop_gender_age['생활인구_남'] / tot_sx_pop_gender_age['생활인구'] * 100).round(1)
tot_sx_pop_gender_age['생활인구_여_비율'] = (tot_sx_pop_gender_age['생활인구_여'] / tot_sx_pop_gender_age['생활인구'] * 100).round(1)
tot_sx_pop_gender_age.to_csv('../P3_Result/01.CSV/1.시흥시_생활인구패턴/전체연도_월평균_성별비율_총생활인구_0922.csv',index=False ,encoding='cp949')
#tot_sx_pop_gender_age.to_csv('../P3_Result/01.CSV/1.시흥시_생활인구패턴/전체연도_월별_성별비율_총생활인구_0922.csv',index=False ,encoding='cp949')

tot_sx_pop_gender_age

In [ ]:
## 연령대별 생활인구 비교 _2023.04기준

#2023.04 기준 주민등록인구 5세별 남여 인구 데이터
region_pop_10 = pd.read_csv('../P1_Data/02.Pretreatment/주민등록인구_10세별_202304기준(수작업).csv',encoding='cp949')
region_pop_10

In [ ]:
pop_gender_age10 = pop_gender_age.copy()
pop_gender_age10 = pop_gender_age10[['etl_ym', 'M00_09', 'M10_19', 'M20_29', 'M30_39','M40_49', 'M50_59', 'M60_up',
                                    'F00_09', 'F10_19', 'F20_29', 'F30_39','F40_49', 'F50_59', 'F60_up']]
pop_gender_age10 = pop_gender_age10[pop_gender_age10['etl_ym'] == 202304.0]
pop_gender_age10

In [ ]:
pop_gender_age10_202304 = pop_gender_age10.groupby('etl_ym').sum().reset_index()

gender_age = ['M00_09', 'M10_19', 'M20_29', 'M30_39','M40_49', 'M50_59', 'M60_up', 'F00_09', 'F10_19', 'F20_29', 'F30_39','F40_49', 'F50_59', 'F60_up']
pop_gender_age10_202304[gender_age] = (pop_gender_age10_202304[gender_age].div(30)).round(1)
pop_gender_age10_202304['생활인구'] = (pop_gender_age10_202304[gender_age].sum(axis=1)).round(1)

#pop_gender_age10_202304.to_csv('../P1_Data/02.Pretreatment/시흥시_생활인구_성연령별(10세)_202304기준.csv',index=False,encoding='cp949')

#### 2) 시간대별 생활인구 비교

In [ ]:
pre_pop_time = pop_time.copy()
time = ['0', '1', '2', '3', '4', '5', '6', '7', '8',
       '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20',
       '21', '22', '23']
pre_pop_time['생활인구'] = pre_pop_time[time].sum(axis=1)
pre_pop_time

In [ ]:
#각 연월 시간대별 합계
average_by_year = pre_pop_time.groupby(['Year','Month'])['0', '1', '2', '3', '4', '5', '6', '7', '8',
       '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20',
       '21', '22', '23','생활인구'].sum().round(1).reset_index()
average_by_year.to_csv('../P3_Result/01.CSV/1.시흥시_생활인구패턴/시간대별_생활인구비교(연월)_0922.csv',encoding='cp949')
average_by_year

#### 3) 요일별 생활인구 비교

In [ ]:
day_pop1 = pop_day.copy()

In [ ]:
day = ['Sunday', 'Monday', 'Tuesday','Wednesday', 'Thursday', 'Friday', 'Saturday']
day_pop1['생활인구'] = day_pop1[day].sum(axis=1)
day_pop1

In [ ]:
# 연도별로 그룹화하고 각 요일별 생활인구수의 평균 계산
day_by_year = day_pop1.groupby(['Year','Month'])['Sunday', 'Monday', 'Tuesday','Wednesday', 'Thursday', 'Friday', 'Saturday','생활인구'].sum().round(1).reset_index()
#day_by_year = (day_by_year/30).round(1)
day_by_year.to_csv('../P3_Result/01.CSV/1.시흥시_생활인구패턴/요일별_생활인구비교(연월)_0922.csv', encoding='cp949',index=False)
day_by_year